# 🛍️ لوحة أداء مبيعات التجارة الإلكترونية (Sales Performance Dashboard)
### مشروع A1 — مسار تحليل البيانات (Data Analysis Track)

---
## 🎯 المشكلة التجارية (Business Problem)
إدارة متجر إلكتروني عايزة تعرف **صحّة البيزنس في لمحة**: إيرادات، عدد الطلبات، متوسط قيمة الطلب،
أكتر منتجات/دول بتبيع، والاتجاه الشهري — عشان تاخد قرارات (مخزون، تسويق، توسّع).

**نوع المشكلة:** تحليل وصفي (Descriptive Analytics) + بناء **مؤشرات أداء (KPIs)** ولوحة متابعة.

## 📦 ما الذي يثبته المشروع
**SQL** (تجميع واستعلام) · **Pandas** · حساب **KPIs** · التصوير (Visualization) ·
ولوحة تفاعلية للإنتاج بـ **Streamlit + Plotly** (في `src/app.py`).


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "data_analysis/a1_sales_dashboard"          # مسار المشروع داخل portfolio/
PKGS    = []          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| SQL (GROUP BY, تجميع) | Kleppmann / أي مرجع SQL · McKinney (ch.) | لغة استخراج المؤشرات من البيانات |
| Pandas للتجميع | McKinney — *Python for Data Analysis* (ch.10) | groupby/pivot كبديل/مكمّل لـ SQL |
| **مؤشرات الأداء (KPIs)** | أي مرجع Business Analytics | Revenue · AOV · Repeat-rate · إلخ |
| التصوير الفعّال | Knaflic — *Storytelling with Data* | اللوحة لازم توصّل القصة بسرعة |
| لوحات تفاعلية | وثائق Streamlit / Plotly | نشر اللوحة لغير التقنيين |

> 🎯 **بيُستخدم في الواقع:** لوحات الإدارة (Executive dashboards)، تقارير المبيعات الأسبوعية، أدوات BI.
> 🛠️ **هنا:** نستخرج المؤشرات بـ **SQL حقيقي** (sqlite داخل الذاكرة) ونرسمها — والنسخة التفاعلية في `src/app.py`.


## 0️⃣ المكتبات وتحميل البيانات

In [2]:
import sqlite3
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); plt.rcParams['figure.dpi'] = 110

df = pd.read_csv('data/ecommerce_transactions.csv', parse_dates=['order_date'])
print('Shape:', df.shape, '| من', df['order_date'].min().date(), 'إلى', df['order_date'].max().date())
df.head(3)

Shape: (15038, 10) | من 2022-07-03 إلى 2023-12-31


,order_id,customer_id,order_date,product_name,category,unit_price,quantity,total_amount,country,payment_method
0,ORD113311,CUST2448,2023-10-13,Monitor,Electronics,1166.93,2,2333.86,Canada,PayPal
1,ORD112626,CUST2379,2023-03-23,Perfume,Beauty,75.84,2,151.68,Australia,Credit Card
2,ORD106393,CUST1671,2023-03-30,T-Shirt,Clothing,176.91,1,176.91,USA,PayPal


## 1️⃣ تحميل البيانات في قاعدة SQL (sqlite في الذاكرة)
بدل ما نعمل كل حاجة بـ pandas، هنحمّل الجدول في **sqlite** ونكتب **SQL حقيقي** — ده اللي بيحصل في
الشغل الفعلي (البيانات في قاعدة بيانات، والمحلل بيكتب استعلامات).

In [3]:
conn = sqlite3.connect(':memory:')
df.to_sql('sales', conn, index=False, if_exists='replace')

def q(sql):
    return pd.read_sql(sql, conn)

print(q("SELECT COUNT(*) AS rows, COUNT(DISTINCT customer_id) AS customers FROM sales"))

    rows  customers
0  15038       1600


## 2️⃣ مؤشرات الأداء الرئيسية (KPIs) — بـ SQL
- **Revenue:** إجمالي الإيرادات · **Orders:** عدد الطلبات · **AOV:** متوسط قيمة الطلب (Average Order Value)
- **Customers:** عدد العملاء · **Repeat-rate:** نسبة العملاء اللي طلبوا أكتر من مرة

In [4]:
kpi = q('''
    SELECT
        ROUND(SUM(total_amount), 0)                    AS revenue,
        COUNT(*)                                       AS orders,
        ROUND(SUM(total_amount) * 1.0 / COUNT(*), 2)   AS aov,
        COUNT(DISTINCT customer_id)                    AS customers
    FROM sales
''').iloc[0]

repeat = q('''
    WITH per_customer AS (
        SELECT customer_id, COUNT(*) AS n FROM sales GROUP BY customer_id
    )
    SELECT ROUND(AVG(n > 1) * 100, 1) AS repeat_rate_pct FROM per_customer
''').iloc[0, 0]

print(f"💰 Revenue       = ${kpi.revenue:,.0f}")
print(f"🧾 Orders        = {kpi.orders:,}")
print(f"📊 AOV           = ${kpi.aov:,.2f}")
print(f"👥 Customers     = {kpi.customers:,}")
print(f"🔁 Repeat rate   = {repeat}%")

💰 Revenue       = $5,882,563
🧾 Orders        = 15,038.0
📊 AOV           = $391.18
👥 Customers     = 1,600.0
🔁 Repeat rate   = 83.5%


## 3️⃣ الاتجاه الشهري للإيرادات (Monthly Trend) — SQL + رسم

In [5]:
monthly = q('''
    SELECT strftime('%Y-%m', order_date) AS month,
           ROUND(SUM(total_amount), 0)   AS revenue,
           COUNT(*)                      AS orders
    FROM sales GROUP BY month ORDER BY month
''')
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly['month'], monthly['revenue'], marker='o')
ax.set_title('Monthly Revenue'); ax.set_ylabel('Revenue ($)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()
monthly.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_3116\2465838770.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()


,month,revenue,orders
0,2022-07,29890.0,87
1,2022-08,78726.0,231
2,2022-09,142647.0,365
3,2022-10,251375.0,622
4,2022-11,268497.0,752


## 4️⃣ التقسيمات: الفئة · أعلى المنتجات · الدول (Breakdowns)

In [6]:
by_cat = q("SELECT category, ROUND(SUM(total_amount),0) AS revenue "
            "FROM sales GROUP BY category ORDER BY revenue DESC")
top_prod = q("SELECT product_name, ROUND(SUM(total_amount),0) AS revenue, SUM(quantity) AS units "
             "FROM sales GROUP BY product_name ORDER BY revenue DESC LIMIT 10")
by_country = q("SELECT country, ROUND(SUM(total_amount),0) AS revenue "
               "FROM sales GROUP BY country ORDER BY revenue DESC")
print('الفئات:'); print(by_cat.to_string(index=False))
print('\nأعلى 5 منتجات:'); print(top_prod.head().to_string(index=False))

الفئات:
      category   revenue
   Electronics 3930691.0
Home & Kitchen  822289.0
      Clothing  597807.0
        Beauty  388705.0
         Books  143070.0

أعلى 5 منتجات:
product_name  revenue  units
  Smartphone 825640.0   1254
  Headphones 797725.0   1230
    Keyboard 793179.0   1228
     Monitor 775439.0   1231
      Laptop 738709.0   1170


## 5️⃣ اللوحة الثابتة (Dashboard) — كل المؤشرات في صورة واحدة
لوحة من 4 لوحات فرعية: الاتجاه الشهري · الإيراد حسب الفئة · أعلى المنتجات · أعلى الدول.

In [7]:
fig, ax = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('🛍️ E-Commerce Sales Dashboard', fontsize=16, fontweight='bold')

ax[0,0].plot(monthly['month'], monthly['revenue'], marker='o', color='#2563eb')
ax[0,0].set_title('Monthly Revenue'); ax[0,0].tick_params(axis='x', rotation=45)

sns.barplot(data=by_cat, y='category', x='revenue', ax=ax[0,1], color='#10b981')
ax[0,1].set_title('Revenue by Category'); ax[0,1].set_ylabel('')

sns.barplot(data=top_prod, y='product_name', x='revenue', ax=ax[1,0], color='#f59e0b')
ax[1,0].set_title('Top 10 Products'); ax[1,0].set_ylabel('')

sns.barplot(data=by_country.head(8), y='country', x='revenue', ax=ax[1,1], color='#ef4444')
ax[1,1].set_title('Top Countries'); ax[1,1].set_ylabel('')

plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()

C:\Users\DELL\AppData\Local\Temp\ipykernel_3116\2520192905.py:16: UserWarning: Glyph 128717 (\N{SHOPPING BAGS}) missing from font(s) Arial.
  plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()
C:\Users\DELL\AppData\Local\Temp\ipykernel_3116\2520192905.py:16: UserWarning: Glyph 65039 (\N{VARIATION SELECTOR-16}) missing from font(s) Arial.
  plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()
C:\Users\DELL\AppData\Local\Temp\ipykernel_3116\2520192905.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()


## 6️⃣ الخلاصة والتوصيات (Conclusion)
- **المؤشرات:** استخرجناها بـ **SQL حقيقي** (sqlite) زي ما بيحصل في الشغل، مش بس pandas.
- **اللوحة:** بتوصّل صحّة البيزنس في صورة واحدة — اتجاه + أعلى فئات/منتجات/دول.
- **قرارات مقترحة:** ركّز المخزون على أعلى المنتجات، استهدف الدول الأعلى إيراداً، واشتغل على رفع
  **Repeat-rate** (ولاء العملاء) لأن جذب عميل جديد أغلى من الاحتفاظ بعميل.
- **للإنتاج:** اللوحة التفاعلية في `src/app.py` (Streamlit + Plotly) فيها فلاتر (دولة/فئة/تاريخ).

> ▶️ **شغّل اللوحة التفاعلية:**
> ```bash
> pip install streamlit plotly
> streamlit run src/app.py
> ```
> ✅ **اللي اتعلمته:** SQL للتجميع، حساب KPIs، التصوير كلوحة، وربطها بأداة تفاعلية.
